## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install timm==0.9.16
!pip install lightning==2.0.0
!pip install pytorch-lightning==2.3.0
!pip install albumentations
!pip install ttach
!pip install opencv-python
!pip install scikit-learn
!pip install torch torchvision torchaudio

## 2. Upload GeoSeg Code

**Option A:** Upload your GeoSeg folder as a zip file  
**Option B:** Clone from GitHub (if available)

In [ ]:
# Option A: Upload GeoSeg.zip and extract
from google.colab import files
import zipfile
import os

print("Please upload GeoSeg.zip file")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f'Extracting {filename}...')
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print('Done!')

!ls -la

## 3. Upload Archive Dataset

Upload your archive.zip file containing Train, Val, and Test folders

In [ ]:
# Upload archive dataset
print("Please upload archive.zip file")
uploaded = files.upload()

for filename in uploaded.keys():
    if 'archive' in filename.lower() and filename.endswith('.zip'):
        print(f'Extracting {filename}...')
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print('Done!')

!ls -la archive/

## 4. Convert Masks

In [ ]:
# Convert masks for all dataset splits
import os

# Change to GeoSeg directory
os.chdir('GeoSeg')

# Convert Train Rural masks
!python tools/loveda_mask_convert.py \
    --mask-dir ../archive/Train/Train/Rural/masks_png \
    --output-mask-dir ../archive/Train/Train/Rural/masks_png_convert

# Convert Train Urban masks
!python tools/loveda_mask_convert.py \
    --mask-dir ../archive/Train/Train/Urban/masks_png \
    --output-mask-dir ../archive/Train/Train/Urban/masks_png_convert

# Convert Val Rural masks
!python tools/loveda_mask_convert.py \
    --mask-dir ../archive/Val/Val/Rural/masks_png \
    --output-mask-dir ../archive/Val/Val/Rural/masks_png_convert

# Convert Val Urban masks
!python tools/loveda_mask_convert.py \
    --mask-dir ../archive/Val/Val/Urban/masks_png \
    --output-mask-dir ../archive/Val/Val/Urban/masks_png_convert

print('\n✓ All masks converted successfully!')

## 5. Create Training Configuration

In [ ]:
# Create config for archive dataset
config_content = '''from torch.utils.data import DataLoader
from geoseg.losses import *
from geoseg.datasets.loveda_dataset import *
from geoseg.models.UNetFormer import UNetFormer
from tools.utils import Lookahead
from tools.utils import process_model_params

# training hparam
max_epoch = 30
ignore_index = len(CLASSES)
train_batch_size = 8  # Increased for GPU
val_batch_size = 8
lr = 6e-4
weight_decay = 0.01
backbone_lr = 6e-5
backbone_weight_decay = 0.01
num_classes = len(CLASSES)
classes = CLASSES

weights_name = "unetformer-archive-colab"
weights_path = "trained_models/archive/{}".format(weights_name)
test_weights_name = "last"
log_name = 'archive/{}'.format(weights_name)
monitor = 'val_mIoU'
monitor_mode = 'max'
save_top_k = 1
save_last = True
check_val_every_n_epoch = 1
pretrained_ckpt_path = None
gpus = 'auto'
resume_ckpt_path = None

#  define the network
net = UNetFormer(num_classes=num_classes)

# define the loss
loss = UnetFormerLoss(ignore_index=ignore_index)
use_aux_loss = True

# define the dataloader
def get_training_transform():
    train_transform = [
        albu.HorizontalFlip(p=0.5),
        albu.Normalize()
    ]
    return albu.Compose(train_transform)

def train_aug(img, mask):
    crop_aug = Compose([RandomScale(scale_list=[0.75, 1.0, 1.25, 1.5], mode='value'),
                        SmartCropV1(crop_size=512, max_ratio=0.75, ignore_index=ignore_index, nopad=False)])
    img, mask = crop_aug(img, mask)
    img, mask = np.array(img), np.array(mask)
    aug = get_training_transform()(image=img.copy(), mask=mask.copy())
    img, mask = aug['image'], aug['mask']
    return img, mask

def get_val_transform():
    val_transform = [
        albu.Normalize()
    ]
    return albu.Compose(val_transform)

def val_aug(img, mask):
    img, mask = np.array(img), np.array(mask)
    aug = get_val_transform()(image=img.copy(), mask=mask.copy())
    img, mask = aug['image'], aug['mask']
    return img, mask

# Updated paths to point to archive data
train_dataset = LoveDATrainDataset(transform=train_aug, data_root='../archive/Train/Train')
val_dataset = LoveDATrainDataset(data_root='../archive/Val/Val', mosaic_ratio=0.0, transform=val_aug)
test_dataset = LoveDATestDataset(data_root='../archive/Test/Test')

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=train_batch_size,
                          num_workers=2,
                          pin_memory=True,
                          shuffle=True,
                          drop_last=True)

val_loader = DataLoader(dataset=val_dataset,
                        batch_size=val_batch_size,
                        num_workers=2,
                        shuffle=False,
                        pin_memory=True,
                        drop_last=False)

# define the optimizer
layerwise_params = {"backbone.*": dict(lr=backbone_lr, weight_decay=backbone_weight_decay)}
net_params = process_model_params(net, layerwise_params=layerwise_params)
base_optimizer = torch.optim.AdamW(net_params, lr=lr, weight_decay=weight_decay)
optimizer = Lookahead(base_optimizer)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epoch, eta_min=1e-6)
'''

# Write config file
os.makedirs('config/loveda', exist_ok=True)
with open('config/loveda/unetformer_archive_colab.py', 'w') as f:
    f.write(config_content)

print('✓ Configuration file created!')

## 6. Start Training

This will train for 30 epochs. Training time on GPU: ~3-4 hours

In [ ]:
# Start training
!python train_supervision.py -c config/loveda/unetformer_archive_colab.py

## 7. Download Trained Model

In [ ]:
# Zip and download trained model
import shutil

# Zip the trained models
shutil.make_archive('trained_models', 'zip', 'trained_models')

# Download
from google.colab import files
files.download('trained_models.zip')

print('✓ Model downloaded!')

## 8. Monitor Training (Optional)

Check training logs and metrics

In [ ]:
# View training logs
!tail -50 lightning_logs/version_*/metrics.csv

## 9. Check Model Checkpoints

Verify if trained models have been saved

In [ ]:
import os
from pathlib import Path

# Check for saved models
weights_dir = Path("GeoSeg/trained_models/archive")
logs_dir = Path("GeoSeg/lightning_logs/archive")

print("=" * 70)
print("CHECKING FOR SAVED MODEL CHECKPOINTS")
print("=" * 70)

# Check weights directory
print(f"\n📁 Checking weights directory: {weights_dir}")
if weights_dir.exists():
    print(f"   ✓ Directory exists")
    ckpt_files = list(weights_dir.glob("*.ckpt"))
    if ckpt_files:
        print(f"   ✓ Found {len(ckpt_files)} checkpoint file(s):\n")
        for ckpt in sorted(ckpt_files):
            size_mb = ckpt.stat().st_size / (1024 * 1024)
            print(f"      📦 {ckpt.name}")
            print(f"         Size: {size_mb:.2f} MB")
            print()
    else:
        print(f"   ✗ No .ckpt files found - training may not have started yet")
else:
    print(f"   ✗ Directory does not exist - training has not started")

# Check logs directory
print(f"\n📊 Checking logs directory: {logs_dir}")
if logs_dir.exists():
    print(f"   ✓ Directory exists")
    subdirs = sorted([d for d in logs_dir.iterdir() if d.is_dir()])
    if subdirs:
        print(f"   ✓ Found {len(subdirs)} log folder(s):\n")
        for subdir in subdirs:
            csv_files = list(subdir.glob("*.csv"))
            print(f"      📋 {subdir.name}")
            print(f"         CSV files: {len(csv_files)}")
            
            # Check metrics.csv for training progress
            metrics_csv = subdir / "metrics.csv"
            if metrics_csv.exists():
                with open(metrics_csv, 'r') as f:
                    lines = f.readlines()
                    if len(lines) > 1:
                        # Get header and last line
                        header = lines[0].strip()
                        last_line = lines[-1].strip()
                        print(f"         Epochs logged: {len(lines) - 1}")
                        print(f"         Last entry: {last_line[:80]}...")
            print()
    else:
        print(f"   ✗ No log folders found")
else:
    print(f"   ✗ Directory does not exist")

# Check training status
print(f"\n🔍 Training Status:")
if weights_dir.exists():
    last_ckpt = weights_dir / "last.ckpt"
    if last_ckpt.exists():
        print(f"   ✓ last.ckpt exists - at least 1 epoch completed")
        size_mb = last_ckpt.stat().st_size / (1024 * 1024)
        print(f"   📦 Size: {size_mb:.2f} MB")
        
        # Count best model checkpoints
        best_ckpts = [f for f in weights_dir.glob("*.ckpt") if f.name != "last.ckpt"]
        if best_ckpts:
            print(f"   ✓ Best model saved: {best_ckpts[0].name}")
        else:
            print(f"   ⚠ No best model checkpoint yet")
    else:
        print(f"   ⚠ last.ckpt not found - training may still be in progress or hasn't started")
else:
    print(f"   ✗ Weights directory not created yet - training has not started")

print("\n" + "=" * 70)
print("✓ Check complete!")
print("=" * 70)